In [1]:
import numpy as np
import h5py
from scipy.signal import butter, filtfilt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
# =========================================================
# CONFIGURATION
# =========================================================
DATA_DIR = "padded_EEG.h5" # input file: padded EEG trials
OUT_FILE = "EEG_DE_data.h5" # output file: extracted DE features

sfreq = 200                     # sampling frequency in Hz
window_sec = 1.0                # window duration in seconds
overlap = 0                     # overlap fraction between windows (0 = no overlap)
n_bins_hist = 30                # number of histogram bins for entropy estimation

# frequency bands for EEG decomposition (in Hz)
bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}

channels = list(range(62))  # use all 62 EEG channels
band_names = list(bands.keys()) # list of band names for metadata

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def bandpass_filter(signal, sfreq, f_low, f_high, order=4):
    """
    Apply a zero-phase Butterworth bandpass filter to a 1D signal.

    Parameters
    ----------
    signal : array-like
        Input EEG signal (1D).
    sfreq : float
        Sampling frequency in Hz.
    f_low : float
        Lower cutoff frequency in Hz.
    f_high : float
        Upper cutoff frequency in Hz.
    order : int
        Filter order (default: 4).

    Returns
    -------
    np.ndarray
        Filtered signal with the same length as input.
    """
    nyq = sfreq / 2.0
    b, a = butter(order, [f_low / nyq, f_high / nyq], btype="band")
    return filtfilt(b, a, signal) # filtfilt applies zero-phase filtering

def differential_entropy_hist(x, n_bins=30, eps=1e-12):
    """
    Estimate differential entropy (DE) from a signal using a normalized histogram.

    Unlike the parametric Gaussian estimate h(X) = 0.5 * log(2*pi*e*sigma^2),
    this method makes no assumptions about the underlying distribution,
    making it more robust for non-Gaussian EEG signals.

    The correction term log(dx) accounts for the discretization of
    the continuous integral, ensuring a valid approximation of h(X).

    Parameters
    ----------
    x : array-like
        Input signal window (1D).
    n_bins : int
        Number of histogram bins (default: 30).
    eps : float
        Small constant added to avoid log(0) (default: 1e-12).

    Returns
    -------
    float
        Estimated differential entropy value.
    """
    
    x = np.asarray(x)
    hist, bin_edges = np.histogram(x, bins=n_bins, density=True)
    hist = hist + eps                   # avoid log(0)
    hist = hist / np.sum(hist)          # normalize to probability distribution
    dx = bin_edges[1] - bin_edges[0]    # bin width for discretization correction
    entropy = -np.sum(hist * np.log(hist)) + np.log(dx)
    return entropy

def compute_n_windows(n_samples, window_size, step):
    """
    Compute the number of non-overlapping windows for a given signal length.

    Parameters
    ----------
    n_samples : int
        Total number of samples in the signal.
    window_size : int
        Number of samples per window.
    step : int
        Number of samples between consecutive window starts.

    Returns
    -------
    int
        Number of windows.
    """
    return (n_samples - window_size) // step + 1

# =========================================================
# FEATURE EXTRACTION
# =========================================================
def extract_features(eeg, window_size, step, n_channels, n_bands):
    """
    Extract differential entropy (DE) features from a single EEG trial.

    For each channel, the signal is:
      1. DC-corrected by subtracting the mean
      2. Bandpass-filtered into each frequency band
      3. Segmented into non-overlapping windows
      4. DE is estimated per window per band using a histogram

    Parameters
    ----------
    eeg : np.ndarray
        EEG trial of shape (n_channels, n_samples).
    window_size : int
        Number of samples per window.
    step : int
        Step size between consecutive windows.
    n_channels : int
        Number of EEG channels.
    n_bands : int
        Number of frequency bands.

    Returns
    -------
    np.ndarray
        Feature array of shape (n_channels, n_windows, n_bands).
    """
    
    n_samples = eeg.shape[1]
    n_windows = compute_n_windows(n_samples, window_size, step)

    # pre-allocate feature array for efficiency
    features = np.zeros((n_channels, n_windows, n_bands), dtype=np.float32)


    for ch_i in range(n_channels):
        sig = eeg[ch_i].astype(np.float64)

        # remove DC offset by subtracting the channel mean
        sig = sig - np.mean(sig)

        # apply bandpass filter to the full signal for each frequency band
        # filtering the full signal before windowing avoids edge artifacts
        filtered_signals = []
        for f_low, f_high in bands.values():
            filtered = bandpass_filter(sig, sfreq, f_low, f_high)
            filtered_signals.append(filtered)

        # slide windows over the filtered signals and compute DE per window
        starts = range(0, n_samples - window_size + 1, step)
        for w_i, start in enumerate(starts):
            for b_i, band_sig in enumerate(filtered_signals):
                window = band_sig[start:start + window_size]
                features[ch_i, w_i, b_i] = differential_entropy_hist(window, n_bins_hist)

    return features

# =========================================================
# DATASET PROCESSING
# =========================================================
print("=" * 60)
print("EEG PREPROCESSING - FEATURE EXTRACTION")
print("=" * 60)

with h5py.File(DATA_DIR, "r") as f:
    data = f["data"]          # shape: (N, n_channels, n_samples)
    labels = f["labels"][:]

    N = data.shape[0]            # number of trials
    n_channels = data.shape[1]     # number of channels
    n_samples = data.shape[2]     # number of time samples per trial

    # compute window and step size in samples
    window_size = int(window_sec * sfreq)        # e.g. 1.0 * 200 = 200 samples
    step = int(window_size * (1 - overlap))      # step = window_size when overlap = 0
    step = max(step, 1)                          # ensure step is at least 1

    n_windows_total = compute_n_windows(n_samples, window_size, step)
    n_bands = len(bands)

    print(f"Data loaded: {N} trials, {n_channels} channels, {n_samples} samples/channel")
    print(f"Windows: {n_windows_total} (size {window_size}, step {step})")
    print(f"Bands: {n_bands}")

    # pre-allocate output arrays for all trials
    X = np.zeros((N, n_channels, n_windows_total, n_bands), dtype=np.float32)
    y = np.zeros(N, dtype=np.int64)

    # extract features trial by trial
    for i in tqdm(range(N), desc="Extracting features"):
        X[i] = extract_features(data[i], window_size, step, n_channels, n_bands)
        y[i] = labels[i]

print(f"\nFinal shape: {X.shape}")
print(f"Labels: {y.shape}")

# =========================================================
# VALIDATION AND METADATA
# =========================================================
# check for NaN or infinite values that would corrupt training
if np.any(np.isnan(X)) or np.any(np.isinf(X)):
    print("\nWARNING: NaN or Inf values found in features!")
    print(f"   NaNs: {np.sum(np.isnan(X))}")
    print(f"   Infs: {np.sum(np.isinf(X))}")
else:
    print("\nFeatures contain no invalid values.")

# basic statistics to sanity-check the extracted features
print(f"\nFeature statistics:")
print(f"   Mean:  {np.mean(X):.6f}")
print(f"   Std:   {np.std(X):.6f}")
print(f"   Min:   {np.min(X):.6f}")
print(f"   Max:   {np.max(X):.6f}")

# =========================================================
# SAVE OUTPUT FILE
# =========================================================
with h5py.File(OUT_FILE, "w") as f:
    # save feature array with gzip compression to reduce file size
    f.create_dataset("data", data=X, compression="gzip", compression_opts=4)
    f.create_dataset("labels", data=y)
    f.create_dataset("channels", data=np.array(channels))
    # store band names as byte strings for HDF5 compatibility
    f.create_dataset("band_names", data=np.array(band_names, dtype="S"))

    # store extraction parameters as file-level attributes for reproducibility
    f.attrs["sfreq"] = sfreq
    f.attrs["window_sec"] = window_sec
    f.attrs["overlap"] = overlap
    f.attrs["window_size"] = window_size
    f.attrs["step"] = step
    f.attrs["n_windows"] = n_windows_total
    f.attrs["n_bands"] = n_bands
    f.attrs["entropy_method"] = "histogram"
    f.attrs["n_bins_histogram"] = n_bins_hist
    f.attrs["structure"] = "N x channels x windows x bands"

print(f"\nFile saved successfully: {OUT_FILE}")
print(f"   Approximate size: {X.nbytes / 1e6:.1f} MB")

EEG PREPROCESSING - FEATURE EXTRACTION
Data loaded: 675 trials, 62 channels, 48000 samples/channel
Windows: 240 (size 200, step 200)
Bands: 5


Extracting features: 100%|████████████████████| 675/675 [57:14<00:00,  5.09s/it]



Final shape: (675, 62, 240, 5)
Labels: (675,)

Features contain no invalid values.

Feature statistics:
   Mean:  0.160259
   Std:   8.096159
   Min:   -50.216473
   Max:   11.965197

File saved successfully: EEG_DE_data.h5
   Approximate size: 200.9 MB
